# Turbo 5-SAT Solver — maksimal, adu cepat (versi contoh portfolio 8 solver)

Notebook ini menjalankan solver 5-SAT paralel maksimal untuk tantangan NONCE Genesis (profil `nonce-logic-balanced-1536`: 1.536 variabel, 33.792 klausa, 5 literal per klausa).

Prinsipnya adu cepat: semua solver jalan bersamaan di proses terpisah (satu solver per core), siapa yang dapat jawaban TERVERIFIKASI duluan dia yang menang, sisanya langsung dimatikan. Daftar solver mengikuti contoh portfolio yang kamu kirim (8 solver, dipilih sebanyak jumlah core, minimal 2).

**Cara pakai:**
- **Mode 1 (upload):** jalankan cell install, cell solver, lalu cell upload — pilih file DIMACS (`.cnf`) atau JSON list klausa, solve, salin bitstring hasilnya.
- **Mode 2 (siaga, untuk attempt live):** jalankan cell siaga SEBELUM attempt dimulai. Runtime harus sudah hangat (solver terinstall, notebook siap) karena deadline attempt hanya 900 detik — waktu setup tidak boleh makan deadline.

**Keamanan:** yang dikirim ke notebook ini HANYA klausa (data puzzle). Jangan pernah menempel token, capability, atau signature apa pun di sini. Bitstring hasil solve HARUS diverifikasi lokal terhadap semua klausa sebelum di-submit ke server.

In [ ]:
try:
    import pysat
    print('pysat sudah tersedia')
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-sat'])
    import pysat
    print('pysat berhasil diinstall')

In [ ]:
import os, time, json, multiprocessing as mp
from pysat.solvers import Solver

# Daftar portfolio dari contoh yang kamu kirim, sesuai urutan kekuatannya.
# Dipilih sebanyak jumlah core (minimal 2) supaya benar-benar adu cepat.
SOLVERS = ['cadical195', 'cadical153', 'glucose4', 'maplechrono', 'lingeling', 'minisat22', 'glucose3', 'maplesat']

def load(path):
    txt = open(path).read().strip()
    if txt.startswith('['):
        cls = json.loads(txt)
        n = max(abs(l) for c in cls for l in c)
        return n, cls
    n, cls, cur = 0, [], []
    for line in txt.splitlines():
        line = line.strip()
        if not line or line[0] == 'c':
            continue
        if line[0] == 'p':
            n = int(line.split()[2]); continue
        for t in line.split():
            v = int(t)
            if v == 0:
                if cur: cls.append(cur); cur = []
            else:
                cur.append(v)
    if cur: cls.append(cur)
    return n, cls

def verify_assignment(clauses, assignment):
    if isinstance(assignment, str):
        a = {i + 1: (ch == '1') for i, ch in enumerate(assignment)}
    elif isinstance(assignment, dict):
        a = assignment
    else:
        a = {abs(l): (l > 0) for l in assignment}
    for clause in clauses:
        ok = False
        for lit in clause:
            if a.get(abs(lit), False) == (lit > 0):
                ok = True
                break
        if not ok:
            return False
    return True

def to_bitstring(model, n_vars):
    a = {abs(l): (l > 0) for l in model}
    return ''.join('1' if a.get(i, False) else '0' for i in range(1, n_vars + 1))

def _worker(name, clauses, n_vars, queue):
    try:
        with Solver(name=name, bootstrap_with=clauses) as s:
            if s.solve():
                queue.put((name, True, s.get_model()))
            else:
                queue.put((name, False, None))
    except Exception as e:
        queue.put((name, False, 'error: %s' % e))

def portfolio_solve(clauses, n_vars, time_limit=780):
    """Adu cepat: solver paralel sebanyak core; kembalikan (bitstring, pemenang)
    untuk SAT pertama yang TERVERIFIKASI semua klausanya, atau (None, status)."""
    cores = max(1, os.cpu_count() or 1)
    names = SOLVERS[:max(2, min(len(SOLVERS), cores))]
    print('core terdeteksi: %d -> solver: %s' % (cores, ', '.join(names)))
    ctx = mp.get_context('fork')
    queue = ctx.Queue()
    procs = []
    for name in names:
        p = ctx.Process(target=_worker, args=(name, clauses, n_vars, queue))
        p.start()
        procs.append(p)
    deadline = time.time() + time_limit
    result, winner = None, None
    finished = 0
    while time.time() < deadline and finished < len(procs):
        try:
            name, sat, model = queue.get(timeout=5)
            finished += 1
            if sat and model is not None:
                bits = to_bitstring(model, n_vars)
                if verify_assignment(clauses, bits):
                    result, winner = bits, name
                    break
        except Exception:
            pass
    for p in procs:
        if p.is_alive():
            p.terminate()
    for p in procs:
        p.join(timeout=5)
    if result is not None:
        return result, winner
    return None, ('timeout' if time.time() >= deadline else 'unsat_or_error')

print('solver siap')

In [ ]:
# MODE 1: upload file DIMACS (.cnf) atau JSON list klausa, lalu solve
from google.colab import files
uploaded = files.upload()
cnf_path = list(uploaded.keys())[0]
n_vars, clauses = load(cnf_path)
print('loaded: %d vars, %d clauses' % (n_vars, len(clauses)))
SOLVE_SECONDS = 780
bits, winner = portfolio_solve(clauses, n_vars, time_limit=SOLVE_SECONDS)
if bits:
    print('WINNER: %s | verified: %s' % (winner, verify_assignment(clauses, bits)))
    print('ASSIGNMENT_BITSTRING:')
    print(bits)
else:
    print('tidak ada solusi terverifikasi (%s)' % winner)

In [ ]:
# MODE 2 (SIAGA, untuk attempt live): tunggu file klausa muncul di URL, lalu solve otomatis.
# Jalankan cell ini SEBELUM attempt dimulai. Ganti CLAUSE_URL dengan link direct file
# (.cnf / .json) yang akan diisi begitu fragment terkumpul.
import urllib.request, time
CLAUSE_URL = 'PASTE_URL_FILE_KLAUSA_DI_SINI'
POLL_EVERY = 10
WAIT_SECONDS = 3600
SOLVE_SECONDS = 780
clause_path = '/tmp/clauses.cnf'
waited = 0
while waited < WAIT_SECONDS:
    try:
        urllib.request.urlretrieve(CLAUSE_URL, clause_path)
        if os.path.getsize(clause_path) > 0:
            break
    except Exception:
        pass
    time.sleep(POLL_EVERY)
    waited += POLL_EVERY
else:
    raise SystemExit('file klausa tidak muncul dalam %d detik' % WAIT_SECONDS)
n_vars, clauses = load(clause_path)
print('loaded: %d vars, %d clauses' % (n_vars, len(clauses)))
bits, winner = portfolio_solve(clauses, n_vars, time_limit=SOLVE_SECONDS)
if bits:
    print('WINNER: %s | verified: %s' % (winner, verify_assignment(clauses, bits)))
    print('ASSIGNMENT_BITSTRING:')
    print(bits)
else:
    print('tidak ada solusi terverifikasi (%s)' % winner)

## Catatan

- Jumlah solver paralel = jumlah core runtime (minimal 2). Di VM ini 2 core → cadical195 + cadical153. Di Colab dengan core lebih banyak, solver berikutnya otomatis ikut adu cepat.
- Jangan pernah submit bitstring yang belum lolos `verify_assignment` atas semua klausa.
- Kalau semua solver selesai UNSAT, biarkan attempt expired (tidak ada fee) — jangan submit jawaban yang belum terverifikasi.